# Kernel Visualisation

Visualising the Gram matrix structure, bandwidth sensitivity, and eigenvalue spectra for three standard kernels: Linear, Polynomial, and RBF.

**Reference:** Shawe-Taylor & Cristianini (2004), Chapters 2–3

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from kernels import LinearKernel, PolynomialKernel, RBFKernel

## Dataset

Two Gaussian clusters in $\mathbb{R}^2$, 15 points each. Points are ordered so rows/columns 0–14 belong to cluster A and 15–29 to cluster B — this lets us see block structure in the Gram matrix directly.

In [ ]:
rng    = np.random.default_rng(42)
X_a    = rng.standard_normal((15, 2)) + np.array([ 2.5,  2.5])
X_b    = rng.standard_normal((15, 2)) + np.array([-2.5, -2.5])
X      = np.vstack([X_a, X_b])
labels = np.array([0] * 15 + [1] * 15)

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(*X_a.T, c='steelblue',  label='Cluster A')
ax.scatter(*X_b.T, c='darkorange', label='Cluster B')
ax.legend(); ax.set_aspect('equal'); ax.set_title('Dataset')
plt.tight_layout(); plt.show()

## 1. Gram matrix heatmaps

The Gram matrix $K_{ij} = k(x_i, x_j)$ encodes pairwise similarity. A kernel that captures the cluster structure should show a clear **block pattern**: high values (bright) within clusters, low values (dark) across clusters.

The dashed red line marks the cluster boundary.

In [ ]:
kernels = [
    ('Linear',           LinearKernel()),
    ('Polynomial (d=3)', PolynomialKernel(degree=3, c=1.0)),
    ('RBF (σ=1.0)',      RBFKernel(sigma=1.0)),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (name, kernel) in zip(axes, kernels):
    K  = kernel.matrix(X)
    im = ax.imshow(K, cmap='viridis')
    ax.axhline(14.5, color='red', linewidth=1.5, linestyle='--')
    ax.axvline(14.5, color='red', linewidth=1.5, linestyle='--')
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('Point index'); ax.set_ylabel('Point index')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Gram matrix heatmaps', fontsize=12)
plt.tight_layout(); plt.show()

## 2. Bandwidth sensitivity — effect of σ on RBF

The RBF bandwidth σ controls the *reach* of each point:

- **Small σ** — kernel decays rapidly; each point attends only to its nearest neighbours
- **Large σ** — kernel decays slowly; all points appear mutually similar

Choosing σ correctly is critical in practice — it is typically set via cross-validation.

In [ ]:
sigmas = [0.3, 1.0, 5.0]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, sigma in zip(axes, sigmas):
    K  = RBFKernel(sigma=sigma).matrix(X)
    im = ax.imshow(K, cmap='viridis', vmin=0, vmax=1)
    ax.axhline(14.5, color='red', linewidth=1.5, linestyle='--')
    ax.axvline(14.5, color='red', linewidth=1.5, linestyle='--')
    ax.set_title(f'RBF  σ = {sigma}', fontsize=13)
    ax.set_xlabel('Point index'); ax.set_ylabel('Point index')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Effect of σ on RBF Gram matrix', fontsize=12)
plt.tight_layout(); plt.show()

## 3. Similarity profile

A single row of the Gram matrix — $k(x_0, x_i)$ for all $i$ — shows which points the kernel considers similar to a fixed query point $x_0$ (cluster A).

A kernel that separates the clusters should assign high similarity to points in cluster A (blue bars, left half) and low similarity to points in cluster B (orange bars, right half).

In [ ]:
bar_cols = ['steelblue' if l == 0 else 'darkorange' for l in labels]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, kernel) in zip(axes, kernels):
    sim = kernel.matrix(X)[0]   # first row = similarity of X[0] to all points
    ax.bar(range(len(X)), sim, color=bar_cols, edgecolor='none', alpha=0.85)
    ax.axvline(14.5, color='black', linewidth=1, linestyle='--')
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Point index'); ax.set_ylabel('k(x₀, xᵢ)')
    ax.legend(handles=[
        Patch(color='steelblue',  label='Cluster A'),
        Patch(color='darkorange', label='Cluster B'),
    ], fontsize=9)

plt.suptitle('Similarity profile of X[0] (cluster A) against all points', fontsize=12)
plt.tight_layout(); plt.show()

## 4. PSD verification — eigenvalue spectra

A valid kernel must produce a **positive semi-definite** Gram matrix — all eigenvalues $\geq 0$. Here we plot the full eigenvalue spectrum for each kernel and verify the condition holds.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, kernel) in zip(axes, kernels):
    K    = kernel.matrix(X)
    eigs = np.linalg.eigvalsh(K)   # sorted ascending; real because K is symmetric
    cols = ['firebrick' if e < -1e-8 else 'steelblue' for e in eigs]
    ax.bar(range(len(eigs)), eigs, color=cols, edgecolor='none')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'{name}\nPSD: {kernel.is_psd(X)} | min λ = {eigs.min():.2e}', fontsize=11)
    ax.set_xlabel('Eigenvalue index (sorted ascending)')
    ax.set_ylabel('Eigenvalue')

plt.suptitle('Eigenvalue spectra of Gram matrices (red = negative = invalid)', fontsize=11)
plt.tight_layout(); plt.show()